In [25]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    roc_auc_score
)


# ============================================================
# 1. Load datasets
# ============================================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
test_labels = pd.read_csv("test_labels.csv")


# ============================================================
# 2. Define target columns
# ============================================================

target_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]


# ============================================================
# 3. Prepare training data
# ============================================================

X_train = train["comment_text"].fillna("")

y_train = train[target_columns].astype(int)


# ============================================================
# 4. Prepare test data
# ============================================================

X_test = test["comment_text"].fillna("")

y_test = test_labels[target_columns].copy()


# ============================================================
# 5. Remove unknown test labels (-1)
# ============================================================

valid_rows = (y_test != -1).all(axis=1)

X_test = X_test[valid_rows]
y_test = y_test[valid_rows].astype(int)


print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


# ============================================================
# 6. TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=150000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    strip_accents="unicode"
)


X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)


print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)


# ============================================================
# 7. Train model
# ============================================================

model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        C=4.0,
        solver="liblinear"
    )
)



model.fit(X_train_tfidf, y_train)


# ============================================================
# 8. Predictions
# ============================================================

y_pred = model.predict(X_test_tfidf)


# ============================================================
# 9. Classification report
# ============================================================

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=target_columns,
        zero_division=0
    )
)





# ============================================================
# 10. Overall metrics
# ============================================================

subset_accuracy = accuracy_score(y_test, y_pred)

micro_f1 = f1_score(
    y_test,
    y_pred,
    average="micro"
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)


print("\nOverall Metrics")
print("-------------------------")
print("Subset Accuracy :", subset_accuracy)
print("Micro F1         :", micro_f1)
print("Macro F1         :", macro_f1)
print("Weighted F1      :", weighted_f1)


Training samples: 159571
Test samples: 63978
TF-IDF training shape: (159571, 150000)
TF-IDF test shape: (63978, 150000)

Classification Report:

               precision    recall  f1-score   support

        toxic       0.64      0.72      0.68      6090
 severe_toxic       0.37      0.34      0.36       367
      obscene       0.77      0.63      0.69      3691
       threat       0.59      0.28      0.38       211
       insult       0.75      0.53      0.62      3427
identity_hate       0.68      0.30      0.42       712

    micro avg       0.69      0.61      0.65     14498
    macro avg       0.63      0.47      0.52     14498
 weighted avg       0.70      0.61      0.64     14498
  samples avg       0.06      0.06      0.06     14498


Overall Metrics
-------------------------
Subset Accuracy : 0.8952452405514395
Micro F1         : 0.6475197725698874
Macro F1         : 0.524777009886299
Weighted F1      : 0.6424408606372656


In [26]:
import joblib
import os

# ============================================================
# Save model and TF-IDF vectorizer
# ============================================================

os.makedirs("saved_model", exist_ok=True)

# Save trained One-vs-Rest Logistic Regression model
joblib.dump(
    model,
    "saved_model/toxicity_model.pkl"
)

# Save fitted TF-IDF vectorizer
joblib.dump(
    vectorizer,
    "saved_model/tfidf_vectorizer.pkl"
)

print("Model and vectorizer saved successfully!")

print("Model path     : saved_model/toxicity_model.pkl")
print("Vectorizer path: saved_model/tfidf_vectorizer.pkl")

Model and vectorizer saved successfully!
Model path     : saved_model/toxicity_model.pkl
Vectorizer path: saved_model/tfidf_vectorizer.pkl
